In [1]:
import numpy as np
import pandas as pd

# 1. Set the seed so our "random" marketplace is reproducible
np.random.seed(42)
n_trips = 10000

# 2. Generate the Confounders (X)
# These are the environmental variables we observe
# weather_severity: 0 is sunny, 5 is a torrential downpour
weather_severity = np.random.uniform(0, 5, n_trips)
# time_of_day: 0 to 24 hours
time_of_day = np.random.uniform(0, 24, n_trips)

# 3. Generate the Treatment: Surge Price (T)
# Here is the confounding! Our legacy pricing algorithm automatically 
# raises the price when the weather is bad (to get more drivers on the road).
# We also add a bit of random noise (the algorithm isn't perfectly deterministic).
surge_price = 1.0 + (0.8 * weather_severity) + np.random.normal(0, 0.2, n_trips)

# Apply our strict business constraints (surge must be between 1.0 and 4.0)
surge_price = np.clip(surge_price, 1.0, 4.0)

# 4. HARDCODE THE TRUE CAUSAL ELASTICITY (The Ground Truth)
# This is the secret parameter we are trying to uncover later!
TRUE_BETA = -2.0  # For every 1.0x increase in surge, demand drops by exactly 2 units

# 5. Generate the Outcome: Rider Demand (Y)
# The "Physics" of our synthetic city:
# - Base demand is 15 riders.
# - Bad weather increases demand (people don't want to walk in the rain).
# - High surge price DECREASES demand (our true beta).
base_demand = 15
demand = (
    base_demand 
    + (4.0 * weather_severity)       # The Confounding Effect: Rain makes demand spike
    + (TRUE_BETA * surge_price)      # The Causal Effect: Price makes demand drop
    + np.random.normal(0, 1.0, n_trips) # Aleatoric uncertainty (random human noise)
)

# 6. Package it into a neat DataFrame
df = pd.DataFrame({
    'weather': weather_severity,
    'time': time_of_day,
    'price': surge_price,
    'demand': demand
})

print(f"Synthetic Marketplace Generated with {n_trips} trips.")
print(df.head())

Synthetic Marketplace Generated with 10000 trips.
    weather       time     price     demand
0  1.872701   8.967380  2.337400  16.939066
1  4.753572   7.989890  4.000000  25.573769
2  3.659970   4.227694  4.000000  20.452101
3  2.993292  14.574400  3.149251  21.441712
4  0.780093  11.438980  1.966536  13.601384


In [ ]:
import statsmodels.formula.api as smf

# 1. The Naive Approach: Predict Demand using ONLY Price
# We are acting like a standard ML engineer who just wants to find the correlation,
# completely ignoring the environmental context (weather).
naive_model = smf.ols("demand ~ price", data=df).fit()

# 2. Extract the estimated coefficient for 'price'
estimated_beta = naive_model.params['price']

print("=== NAIVE MODEL RESULTS ===")
print(f"Estimated Price Elasticity (Beta): {estimated_beta:.2f}")
print(f"True Causal Elasticity (Beta)  : -2.00")

=== NAIVE MODEL RESULTS ===
Estimated Price Elasticity (Beta): 3.61
True Causal Elasticity (Beta)  : -2.00


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict

# Separate our variables into the DML framework
X = df[['weather', 'time']] # The Confounders (Nuisance)
T = df['price']             # The Treatment
Y = df['demand']            # The Outcome

print("Running Double Machine Learning Orthogonalization...")

# --- STAGE 1: Purify the Outcome (Demand) ---
# Predict Demand using ONLY the confounders (weather, time)
model_y = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
y_pred = cross_val_predict(model_y, X, Y, cv=5)

# Calculate the Residuals: The "Surprise" in demand
y_res = Y - y_pred 


# --- STAGE 2: Purify the Treatment (Price) ---
# Predict Price using ONLY the confounders (weather, time)
model_t = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
t_pred = cross_val_predict(model_t, X, T, cv=5)

# Calculate the Residuals: The "Surprise" in price
t_res = T - t_pred 


# --- STAGE 3: The Final Causal Regression ---
# Regress the unexplained demand on the unexplained price
# We set fit_intercept=False because residuals are already centered around zero
final_model = LinearRegression(fit_intercept=False)
final_model.fit(t_res.values.reshape(-1, 1), y_res)

dml_beta = final_model.coef_[0]

print("=== DOUBLE ML RESULTS ===")
print(f"DML Estimated Elasticity (Beta): {dml_beta:.2f}")
print(f"True Causal Elasticity (Beta)  : -2.00")

Running Double Machine Learning Orthogonalization...
=== DOUBLE ML RESULTS ===
DML Estimated Elasticity (Beta): -1.92
True Causal Elasticity (Beta)  : -2.00


In [ ]:
# You would pip install causalml first
from causalml.inference.meta import BaseRRegressor
from xgboost import XGBRegressor

# 1. Initialize the R-Learner
# We pass XGBoost as the engine for both the residual extraction (Stage 1 & 2)
# and the final causal effect modeling (Stage 3).
learner_r = BaseRRegressor(
    learner=XGBRegressor(max_depth=5, random_state=42),
    effect_learner=XGBRegressor(max_depth=3, random_state=42)
)

# 2. Extract our arrays
X = df[['weather', 'time']]  # Confounders
T = df['price']              # Treatment
Y = df['demand']             # Outcome

print("Fitting CausalML R-Learner...")

# 3. Calculate the CATE for every single row in our dataset!
# The fit_predict method handles all the cross-fitting and residual math automatically.
cate_estimates = learner_r.fit_predict(X=X, treatment=T, y=Y)

print("=== CAUSALML RESULTS ===")
print(f"Average Global Elasticity (ATE): {cate_estimates.mean():.2f}")
print(f"Trip 1 Elasticity (Time: {X.iloc[0]['time']:.1f}, Weather: {X.iloc[0]['weather']:.1f}): {cate_estimates[0][0]:.2f}")
print(f"Trip 2 Elasticity (Time: {X.iloc[1]['time']:.1f}, Weather: {X.iloc[1]['weather']:.1f}): {cate_estimates[1][0]:.2f}")

Fitting CausalML R-Learner...


AssertionError: Control group level 0 not found in treatment vector.

In [10]:
import numpy as np
from scipy.optimize import minimize

# ==========================================
# 1. THE PHASE 1 ORACLE (CausalML Output)
# ==========================================
# In reality, this is `learner_r.predict(X)`. 
# We mock the function here so the script runs instantly.
def get_cate(time, weather):
    """Returns the true price elasticity (beta) for a specific context."""
    if time < 12:
        return -0.5  # Morning: Highly inelastic (riders must commute)
    else:
        return -3.0  # Afternoon: Highly elastic (riders will wait or walk)

# ==========================================
# 2. THE CURRENT STATE (The Sensor Data)
# ==========================================
# Imagine a massive concert just let out at 3:00 PM
current_time = 15.0       
current_weather = 1.0     
current_price = 1.0       

current_demand = 100      # 100 riders opening the app
current_supply = 30       # Only 30 drivers in the downtown hexagon
supply_elasticity = 5.0   # Drivers respond to surge by driving toward it

# ==========================================
# 3. THE PHASE 2 SOLVER (The Control Loop)
# ==========================================
def marketplace_loss(proposed_price):
    """The objective function the optimizer will try to minimize."""
    
    # Step A: The Handshake. Ask the Phase 1 model for the gradient!
    beta_cate = get_cate(current_time, current_weather)
    
    # Step B: The System Dynamics Model
    # How does the proposed price change the state?
    price_change = proposed_price - current_price
    
    predicted_demand = current_demand + (beta_cate * price_change)
    predicted_supply = current_supply + (supply_elasticity * price_change)
    
    # Step C: The Cost
    # We want to penalize any mismatch between supply and demand
    mismatch = predicted_demand - predicted_supply
    return mismatch**2

# ==========================================
# 4. EXECUTE THE OPTIMIZATION
# ==========================================
# Hard business constraints: Surge cannot drop below 1.0 or exceed 4.0
surge_bounds = [(1.0, 4.0)]

print("Marketplace Imbalance Detected: 100 Riders vs 30 Drivers.")
print("Running Constrained Optimization Solver...\n")

# We use L-BFGS-B, a standard optimization algorithm for bounded problems
result = minimize(
    marketplace_loss, 
    x0=[1.0],           # Initial guess (start at base price)
    bounds=surge_bounds # Enforce the hard caps!
)

optimal_surge = result.x[0]

# Calculate the projected final state
final_beta = get_cate(current_time, current_weather)
final_demand = current_demand + (final_beta * (optimal_surge - 1.0))
final_supply = current_supply + (supply_elasticity * (optimal_surge - 1.0))

print("=== OPTIMIZATION COMPLETE ===")
print(f"Optimal Surge Price Deployed : {optimal_surge:.2f}x")
print(f"Projected Cleared Demand     : {final_demand:.0f} riders")
print(f"Projected Available Supply   : {final_supply:.0f} drivers")
print(f"Projected Mismatch           : {final_demand - final_supply:.0f}")

Marketplace Imbalance Detected: 100 Riders vs 30 Drivers.
Running Constrained Optimization Solver...

=== OPTIMIZATION COMPLETE ===
Optimal Surge Price Deployed : 4.00x
Projected Cleared Demand     : 91 riders
Projected Available Supply   : 45 drivers
Projected Mismatch           : 46


### Dynamic Pricing

Modeling Predictive Control

In [12]:
import numpy as np
from scipy.optimize import minimize

# ==========================================
# 1. THE PHASE 1 ORACLE (Dynamic CATE)
# ==========================================
def get_cate(time):
    """Returns the true price elasticity (beta)."""
    # Let's say we are transitioning from afternoon to evening
    if time < 17.0:
        return -3.0  # 4:00 PM: Highly elastic
    else:
        return -1.5  # 5:00 PM: Less elastic (rush hour starting)

# ==========================================
# 2. THE SYSTEM STATE & FORECASTS
# ==========================================
H = 3 # Our 3-step Prediction Horizon (e.g., three 5-minute windows)
current_time = 16.8 # 4:48 PM

# Baseline forecasts (What happens if price stays at 1.0x?)
# Demand is naturally dropping, supply is naturally flat.
base_demand_forecast = np.array([100, 95, 90]) 
base_supply_forecast = np.array([30, 30, 30])  

supply_elasticity = 15.0 # Drivers are highly responsive to surge
historical_price = 1.0   # The price 5 minutes ago

# ==========================================
# 3. THE MPC SOLVER (Over Horizon H)
# ==========================================
def mpc_loss(price_sequence):
    """
    Evaluates a proposed sequence of 3 future prices: [P_0, P_1, P_2]
    """
    total_loss = 0
    
    # We will track the state evolution
    D = np.zeros(H)
    S = np.zeros(H)
    
    for k in range(H):
        # 1. Get the dynamic causal elasticity for this specific future time step
        future_time = current_time + (k * 0.1) 
        beta_cate = get_cate(future_time)
        
        # 2. SYSTEM DYNAMICS: Calculate State Evolution
        # Demand reacts instantly to the proposed price at step k
        delta_p_demand = price_sequence[k] - 1.0
        D[k] = base_demand_forecast[k] + (beta_cate * delta_p_demand)
        
        # Supply has a DELAY. It reacts to the previous step's price!
        if k == 0:
            # At step 0, supply is arriving based on the historical price
            delta_p_supply = historical_price - 1.0
        else:
            # At step 1 and 2, supply arrives based on the price we set in the prior step
            delta_p_supply = price_sequence[k-1] - 1.0
            
        S[k] = base_supply_forecast[k] + (supply_elasticity * delta_p_supply)
        
        # 3. CALCULATE THE STEP COST
        # Cost A: Marketplace Mismatch
        mismatch_loss = (D[k] - S[k])**2
        
        # Cost B: Price Volatility (Smoothness Constraint)
        # Penalize huge jumps from the previous price
        lambda_penalty = 50.0 
        if k == 0:
            volatility_loss = lambda_penalty * (price_sequence[k] - historical_price)**2
        else:
            volatility_loss = lambda_penalty * (price_sequence[k] - price_sequence[k-1])**2
            
        total_loss += (mismatch_loss + volatility_loss)
        
    return total_loss

# ==========================================
# 4. EXECUTE THE RECEDING HORIZON
# ==========================================
# Hard business constraints: Surge bounded [1.0, 4.0] for all 3 steps
surge_bounds = [(1.0, 4.0), (1.0, 4.0), (1.0, 4.0)]

# Initial guess for the optimizer: Flat 1.5x surge across the board
initial_guess = [1.5, 1.5, 1.5]

print("Running 3-Step MPC Constrained Optimization...")

result = minimize(
    mpc_loss, 
    x0=initial_guess,           
    bounds=surge_bounds,
    method='L-BFGS-B'
)

optimal_trajectory = result.x

print("\n=== MPC TRAJECTORY COMPUTED ===")
print(f"Step 0 (Now)   : {optimal_trajectory[0]:.2f}x")
print(f"Step 1 (+5 min): {optimal_trajectory[1]:.2f}x")
print(f"Step 2 (+10 min): {optimal_trajectory[2]:.2f}x")

print("\n=== THE RECEDING HORIZON RULE ===")
print(f"ACTION: Deploy {optimal_trajectory[0]:.2f}x to the marketplace now.")
print("ACTION: Discard Step 1 and Step 2. Re-run this entire script in 5 minutes.")

Running 3-Step MPC Constrained Optimization...

=== MPC TRAJECTORY COMPUTED ===
Step 0 (Now)   : 4.00x
Step 1 (+5 min): 4.00x
Step 2 (+10 min): 4.00x

=== THE RECEDING HORIZON RULE ===
ACTION: Deploy 4.00x to the marketplace now.
ACTION: Discard Step 1 and Step 2. Re-run this entire script in 5 minutes.


### Discrete Treatment with CausalML

In [13]:
import numpy as np
import pandas as pd
from causalml.inference.meta import BaseRRegressor
from xgboost import XGBRegressor

# ==========================================
# 1. GENERATE THE CORRECTED DISCRETE SANDBOX
# ==========================================
np.random.seed(42)
n_trips = 10000

# Confounders (X)
time_of_day = np.random.uniform(0, 24, n_trips)
weather = np.random.uniform(0, 5, n_trips) # 0 = Sunny, 5 = Storm
X = pd.DataFrame({'time': time_of_day, 'weather': weather})

# The Discrete Treatment (T): 1 if Promo offered, 0 if Standard Price
# CORRECTED LOGIC: Promos are heavily heavily used on sunny days (weather=0) to boost slow markets.
promo_probability = 0.9 - (0.12 * weather) 
promo_probability = np.clip(promo_probability, 0.1, 0.9)
promo_offered = np.random.binomial(1, promo_probability)

# The Ground Truth CATE Function (The Lift)
# If Time < 12 (Morning commuters): Inelastic to promos (Lift = +1.0 ride)
# If Time >= 12 (Afternoon casuals): Highly elastic to promos (Lift = +5.0 rides)
true_cate_lift = np.where(time_of_day < 12, 1.0, 5.0)

# Outcome (Demand / Trips Booked)
# CORRECTED LOGIC: Base demand starts low (10) on sunny days, but spikes heavily (+3.0 per weather unit) in storms.
base_demand = 10 + (3.0 * weather)
demand = base_demand + (true_cate_lift * promo_offered) + np.random.normal(0, 1.0, n_trips)

# ==========================================
# 2. APPLY CAUSALML (Discrete R-Learner)
# ==========================================
print("Fitting discrete R-Learner for Incentives with corrected physics...")

learner_r = BaseRRegressor(
    learner=XGBRegressor(n_estimators=100, max_depth=3, random_state=42),
    effect_learner=XGBRegressor(n_estimators=100, max_depth=3, random_state=42)
)

# Estimate the CATE (Incremental Lift)
estimated_lift = learner_r.fit_predict(X=X, treatment=promo_offered.astype(int), y=demand)

# ==========================================
# 3. EVALUATE THE RECOVERY
# ==========================================
morning_idx = np.where(time_of_day < 12)[0][0]
afternoon_idx = np.where(time_of_day >= 12)[0][0]

print("\n=== INCENTIVE LIFT ESTIMATES (CATE) ===")
print(f"Morning Trip (True Lift: +1.0)   -> Estimated Lift: +{estimated_lift[morning_idx][0]:.2f}")
print(f"Afternoon Trip (True Lift: +5.0) -> Estimated Lift: +{estimated_lift[afternoon_idx][0]:.2f}")

Fitting discrete R-Learner for Incentives with corrected physics...


/home/hieule/research/causal_ml/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/home/hieule/research/causal_ml/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(



=== INCENTIVE LIFT ESTIMATES (CATE) ===
Morning Trip (True Lift: +1.0)   -> Estimated Lift: +1.04
Afternoon Trip (True Lift: +5.0) -> Estimated Lift: +4.93


In [18]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import cross_val_predict

# ==========================================
# 1. THE CORRECTED DISCRETE SANDBOX 
# (Same as our previous successful script)
# ==========================================
np.random.seed(42)
n_trips = 10000

time_of_day = np.random.uniform(0, 24, n_trips)
weather = np.random.uniform(0, 5, n_trips) 
X = pd.DataFrame({'time': time_of_day, 'weather': weather})

promo_probability = 0.9 - (0.12 * weather) 
promo_probability = np.clip(promo_probability, 0.1, 0.9)
promo_offered = np.random.binomial(1, promo_probability) # Treatment (T)

true_cate_lift = np.where(time_of_day < 12, 1.0, 5.0)

base_demand = 10 + (3.0 * weather)
demand = base_demand + (true_cate_lift * promo_offered) + np.random.normal(0, 1.0, n_trips) # Outcome (Y)

Y = demand
T = promo_offered

print("Executing Discrete R-Learner Math from Scratch...\n")

# ==========================================
# 2. STAGE 1: THE OUTCOME NUISANCE (Regression)
# ==========================================
# Predict expected demand based strictly on weather and time.
model_y = XGBRegressor(n_estimators=100, max_depth=3, random_state=42)
y_expected = cross_val_predict(model_y, X, Y, cv=5)

# Calculate Outcome Residuals (The surprise in demand)
y_res = Y - y_expected

# ==========================================
# 3. STAGE 2: THE TREATMENT NUISANCE (Propensity Score)
# ==========================================
# CRITICAL CHANGE: Because T is binary [0, 1], we use a Classifier!
# We want to predict the probability that T=1.
model_t = XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric='logloss')

# We use method='predict_proba' to get the continuous probability (Propensity Score)
# [:, 1] grabs the probability of the positive class (T=1)
t_expected = cross_val_predict(model_t, X, T, cv=5, method='predict_proba')[:, 1]

# Calculate Treatment Residuals (The random surprise in who got a promo)
t_res = T - t_expected

# ==========================================
# 4. STAGE 3: THE EFFECT MODEL (The Algebraic Hack)
# ==========================================
# We use the exact same weighting trick to force XGBoost to minimize the R-Loss!
cate_target = y_res / t_res
cate_weights = t_res**2

# Train the final non-linear model to learn the Lift function: tau(X)
cate_model = XGBRegressor(n_estimators=100, max_depth=3, random_state=42)
cate_model.fit(X, cate_target, sample_weight=cate_weights)

# ==========================================
# 5. EVALUATE THE RECOVERY
# ==========================================
# Let's test the exact same morning and afternoon trips
morning_idx = np.where(time_of_day < 12)[0][0]
afternoon_idx = np.where(time_of_day >= 12)[0][0]

test_X = X.iloc[[morning_idx, afternoon_idx]]
predicted_lifts = cate_model.predict(test_X)

print("=== INCENTIVE LIFT ESTIMATES (FROM SCRATCH) ===")
print(f"Morning Trip (True Lift: +1.0)   -> Estimated Lift: +{predicted_lifts[0]:.2f}")
print(f"Afternoon Trip (True Lift: +5.0) -> Estimated Lift: +{predicted_lifts[1]:.2f}")

Executing Discrete R-Learner Math from Scratch...

=== INCENTIVE LIFT ESTIMATES (FROM SCRATCH) ===
Morning Trip (True Lift: +1.0)   -> Estimated Lift: +1.11
Afternoon Trip (True Lift: +5.0) -> Estimated Lift: +4.84
